In [129]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,KFold
from sklearn.metrics import accuracy_score,r2_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from tqdm import tqdm
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv(r"E:\DataAnalytics\Cases\Vehicle Silhouettes\Vehicle.csv")
df.head()

,Comp,Circ,D.Circ,Rad.Ra,Pr.Axis.Ra,Max.L.Ra,Scat.Ra,Elong,Pr.Axis.Rect,Max.L.Rect,Sc.Var.Maxis,Sc.Var.maxis,Ra.Gyr,Skew.Maxis,Skew.maxis,Kurt.maxis,Kurt.Maxis,Holl.Ra,Class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


In [3]:
X,y=df.drop('Class',axis=1),df['Class']
le=LabelEncoder()
le_y=le.fit_transform(y)
kfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=26)


In [4]:
solvers = ['lbfgs', 'saga', 'sag', 'newton-cg', 'newton-cholesky']
from tqdm import tqdm
Cs=np.linspace(0.01,3,5)
scores=[]

for l in tqdm(solvers):
    for c in Cs:
        lr=LogisticRegression(solver=l,C=c)
        results=cross_val_score(lr,X,le_y,cv=kfold,scoring='neg_log_loss') # default scoring='accuracy_score'
        scores.append([l,c,results.mean()])
df_scores=pd.DataFrame(scores,columns=['Solvers','C_value','log_loss'])
df_scores.sort_values(by='log_loss',ascending=False).head()

100%|██████████| 5/5 [00:22<00:00,  4.48s/it]


,Solvers,C_value,log_loss
16,newton-cg,0.7575,-0.414607
17,newton-cg,1.5050,-0.421581
18,newton-cg,2.2525,-0.427825
19,newton-cg,3.0000,-0.431223
21,newton-cholesky,0.7575,-0.432451


In [5]:
from sklearn.model_selection import GridSearchCV

In [6]:
lr=LogisticRegression()
params={'solver' : ['lbfgs', 'saga', 'sag', 'newton-cg', 'newton-cholesky'],
       'C':np.linspace(0.01,3,5)
       }
gcv=GridSearchCV(lr,param_grid=params,cv=kfold,scoring='neg_log_loss')
gcv.fit(X,le_y)
print('Best params:',gcv.best_params_)
print('Best score:',gcv.best_score_)

Best params: {'C': np.float64(0.7575000000000001), 'solver': 'newton-cg'}
Best score: -0.41460687009627517


Exercise 

In [7]:
df=pd.read_csv(r"E:\DataAnalytics\Cases\Concrete_Strength\Concrete_Data.csv")
df.head()


,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [14]:
X,y=df.drop('Strength',axis=1),df['Strength']
kfold=KFold(n_splits=5,shuffle=True,random_state=26)
from xgboost import XGBRegressor
xgr=XGBRegressor(random_state=26)
params={'max_depth' : [None,2,3,4],
       'learning_rate':np.linspace(0.01,0.9,10),
        'n_estimators':[25,50,100]
       }
gcv=GridSearchCV(xgr,param_grid=params,cv=kfold,scoring='r2')
gcv.fit(X,y)
print('Best params:',gcv.best_params_)
print('Best score:',gcv.best_score_)

Best params: {'learning_rate': np.float64(0.6033333333333333), 'max_depth': 3, 'n_estimators': 100}
Best score: 0.935401012517735


In [15]:
from sklearn.model_selection import RandomizedSearchCV
params={'max_depth' : [None,2,3,4,5,6,7],
       'learning_rate':np.linspace(0.1,0.9,10),
        'n_estimators':[1,5,10,25,50,100]
       }
rcv=RandomizedSearchCV(xgr,param_distributions=params,cv=kfold,scoring='r2')
rcv.fit(X,y)
print('Best params:',gcv.best_params_)
print('Best score:',gcv.best_score_)

Best params: {'learning_rate': np.float64(0.6033333333333333), 'max_depth': 3, 'n_estimators': 100}
Best score: 0.935401012517735


In [16]:
test=pd.read_csv(r"E:\DataAnalytics\Cases\Concrete_Strength\testConcrete.csv")
test.head()

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age
0,495,120,0,155,5,866,884,75
1,262,129,0,271,2,808,787,174
2,201,48,1,215,5,807,839,113
3,329,141,0,286,1,881,823,229
4,354,14,0,129,2,839,847,210


In [17]:
pred_df=gcv.predict(test)
pred_df

array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)

In [18]:
best_model=XGBRegressor(**gcv.best_params_,random_state=26)
best_model.fit(X,y)
best_model.predict(test)

array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)

In [19]:
bm=gcv.best_estimator_
bm.predict(test)

array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)

## Exercise with pipe line

In [38]:
df=pd.read_csv(r"E:\DataAnalytics\Cases\Medical_Cost_Personal\insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [39]:
X,y = df.drop('charges',axis=1),df['charges']
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')
ct=make_column_transformer(
    (ohe,make_column_selector(dtype_include='object')),
    ('passthrough',make_column_selector(dtype_include='number')),
    verbose_feature_names_out=False
).set_output(transform='pandas')
xgr=XGBRegressor(random_state=26)
pipe=Pipeline([('OHE',ct),('XGR',xgr)])


In [41]:
kfold=KFold(n_splits=5,shuffle=True,random_state=26)
gcv=GridSearchCV(pipe,param_grid=params,cv=kfold,scoring='r2')
                 
params={'XGR__max_depth' : [None,2,3,4],
       'XGR__learning_rate':np.linspace(0.01,0.9,10),
        'XGR__n_estimators':[25,50,100]
       }

gcv.fit(X,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'XGR__learning_rate': array([0.01 ..., 0.9 ]), 'XGR__max_depth': [None, 2, ...], 'XGR__n_estimators': [25, 50, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for cal

In [42]:
print('Best params:',gcv.best_params_)
print('Best score:',gcv.best_score_)

Best params: {'XGR__learning_rate': np.float64(0.10888888888888888), 'XGR__max_depth': 3, 'XGR__n_estimators': 50}
Best score: 0.8632949136193883


In [44]:
kfold=KFold(n_splits=5,shuffle=True,random_state=26)
gcv=GridSearchCV(pipe,param_grid=params,cv=kfold,scoring='r2',n_jobs=-1)
                 
params={'XGR__max_depth' : [None,2,3,4],
       'XGR__learning_rate':np.linspace(0.01,0.9,10),
        'XGR__n_estimators':[25,50,100]
       }

gcv.fit(X,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'XGR__learning_rate': array([0.01 ..., 0.9 ]), 'XGR__max_depth': [None, 2, ...], 'XGR__n_estimators': [25, 50, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for calla

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 
  
# metadata 
print(cirrhosis_patient_survival_prediction.metadata) 
  
# variable information 
print(cirrhosis_patient_survival_prediction.variables) 


In [105]:
X.head()

,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2,4.0
1,D-penicillamine,20617,F,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6,3.0
2,D-penicillamine,25594,M,N,N,N,S,1.4,176,3.48,210,516.0,96.10,55,151,12.0,4.0
3,D-penicillamine,19994,F,N,Y,Y,S,1.8,244,2.54,64,6121.8,60.63,92,183,10.3,4.0
4,Placebo,13918,F,N,Y,Y,N,3.4,279,3.53,143,671.0,113.15,72,136,10.9,3.0


In [106]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Drug           313 non-null    str    
 1   Age            418 non-null    int64  
 2   Sex            418 non-null    str    
 3   Ascites        313 non-null    str    
 4   Hepatomegaly   313 non-null    str    
 5   Spiders        313 non-null    str    
 6   Edema          418 non-null    str    
 7   Bilirubin      418 non-null    float64
 8   Cholesterol    312 non-null    str    
 9   Albumin        418 non-null    float64
 10  Copper         312 non-null    str    
 11  Alk_Phos       312 non-null    float64
 12  SGOT           312 non-null    float64
 13  Tryglicerides  312 non-null    str    
 14  Platelets      411 non-null    str    
 15  Prothrombin    416 non-null    float64
 16  Stage          412 non-null    float64
dtypes: float64(6), int64(1), str(10)
memory usage: 55.6 KB


In [114]:
X=X.replace('NaNN',np.nan)
X['Cholesterol']=pd.to_numeric(X['Cholesterol'],errors='coerce')
X['Copper']=pd.to_numeric(X['Copper'],errors='coerce')
X['Tryglicerides']=pd.to_numeric(X['Tryglicerides'],errors='coerce')
X['Platelets']=pd.to_numeric(X['Platelets'],errors='coerce')

In [115]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Drug           312 non-null    str    
 1   Age            418 non-null    int64  
 2   Sex            418 non-null    str    
 3   Ascites        312 non-null    str    
 4   Hepatomegaly   312 non-null    str    
 5   Spiders        312 non-null    str    
 6   Edema          418 non-null    str    
 7   Bilirubin      418 non-null    float64
 8   Cholesterol    284 non-null    float64
 9   Albumin        418 non-null    float64
 10  Copper         310 non-null    float64
 11  Alk_Phos       312 non-null    float64
 12  SGOT           312 non-null    float64
 13  Tryglicerides  282 non-null    float64
 14  Platelets      407 non-null    float64
 15  Prothrombin    416 non-null    float64
 16  Stage          412 non-null    float64
dtypes: float64(10), int64(1), str(6)
memory usage: 55.6 KB


In [116]:
imp1=SimpleImputer()
imp2=SimpleImputer(strategy='constant',fill_value='unknown')

ct1=make_column_transformer(
    (imp1,make_column_selector(dtype_include='number')),
    (imp2,make_column_selector(dtype_include='object')),
    verbose_feature_names_out=False
).set_output(transform='pandas')


In [ ]:
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
encoder=LabelEncoder()
y_en=encoder.fit_transform(y)

ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')



ct2=make_column_transformer(
    (ohe,make_column_selector(dtype_include='object')),
    ('passthrough',make_column_selector(dtype_include='number')),
    verbose_feature_names_out=False
).set_output(transform='pandas')

st_scl=StandardScaler()
mm_scl=MinMaxScaler()

knn=KNeighborsClassifier()

pipe=Pipeline([('IMP',ct1),('OHE',ct2),('SCL',None),('KNN',knn)])
pipe.get_params()

In [136]:
kfold=KFold(n_splits=5,shuffle=True,random_state=26)
                 
params={'IMP__simpleimputer-1__strategy' : ['mean','median'],
       'IMP__simpleimputer-2__strategy' : ['constant','most_frequent'],
        'SCL':[st_scl,mm_scl,None],
        'KNN__n_neighbors':np.arange(1,11)        
       }

gcv=GridSearchCV(pipe,param_grid=params,cv=kfold,n_jobs=-1,scoring='neg_log_loss')

gcv.fit(X,y_en)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'IMP__simpleimputer-1__strategy': ['mean', 'median'], 'IMP__simpleimputer-2__strategy': ['constant', 'most_frequent'], 'KNN__n_neighbors': array([ 1, 2..., 8, 9, 10]), 'SCL': [StandardScaler(), MinMaxScaler(), ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_log_loss'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit

In [137]:
print('Best params:',gcv.best_params_)
print('Best score:',gcv.best_score_)

Best params: {'IMP__simpleimputer-1__strategy': 'mean', 'IMP__simpleimputer-2__strategy': 'constant', 'KNN__n_neighbors': np.int64(10), 'SCL': None}
Best score: -1.7238268147136417
